In [12]:
import pandas as pd
import numpy as np

In [159]:
from typing import Callable

class MyLineReg():
    def __init__(self, n_iter=100, learning_rate=0.1, metric=None,
                 reg=None, l1_coef=0, l2_coef=0,
                 sgd_sample=None, random_state=42):
        self.n_iter = n_iter
        self.learning_rate = learning_rate
        self.weights = None
        self.metric = metric
        self.reg = reg
        self.l1_coef = l1_coef
        self.l2_coef = l2_coef
        self.sgd_sample = sgd_sample
        self._seed(random_state)
        self.__metric_value = None

    def _seed(self, random_state):
        self.random_state = random_state 
        random.seed(self.random_state)
        
    def __str__(self):
        return f"MyLineReg class: n_iter={self.n_iter}, learning_rate={self.learning_rate}"

    @staticmethod
    def _mse(y_true: pd.Series, y_pred: pd.Series):
        squared_differences = np.square(y_true - y_pred)
        mse = np.mean(squared_differences)
        return mse

    @staticmethod
    def _mae(y_true: pd.Series, y_pred: pd.Series):
        absolute_differrences = np.abs(y_true - y_pred)
        mae = np.mean(absolute_differrences)
        return mae

    @staticmethod
    def _rmse(y_true: pd.Series, y_pred: pd.Series):
        squared_differences = np.square(y_true - y_pred)
        mse = np.mean(squared_differences)
        rmse = np.sqrt(mse)
        return rmse

    @staticmethod
    def _r2(y_true: pd.Series, y_pred: pd.Series):
        y_mean = np.mean(y)
        squared_differences = np.sum(np.square(y_true - y_pred))
        squared_differences_mean = np.sum(np.square(y_true - y_mean))
        r2 = 1 - squared_differences / squared_differences_mean
        return r2

    @staticmethod
    def _mape(y_true: pd.Series, y_pred: pd.Series):
        mape = 100 * np.mean(np.abs(1 - y_pred / y_true))
        return mape

    def _get_metric_value(self, y_true, y_pred):
        if self.metric == 'mae':
            return self._mae(y_true, y_pred)
        if self.metric == 'mse':
            return self._mse(y_true, y_pred)
        if self.metric == 'rmse':
            return self._rmse(y_true, y_pred)
        if self.metric == 'mape':
            return self._mape(y_true, y_pred)
        if self.metric == 'r2':
            return self._r2(y_true, y_pred)
    
    def fit(self, X: pd.DataFrame, y: pd.Series, verbose=False):
        
        if '__bias' in X.columns:
            raise Exception("Column '__bias' exists in X")
        
        X = X.copy()

        X.insert(loc=0, column='__bias', value=1)

        count_features = len(X.columns)
        n = len(X)
            
        self.weights = pd.Series(1, X.columns)

        for i in range(1, self.n_iter + 1):
            
            y_pred = X.dot(self.weights)
            loss = self._mse(y, y_pred)

            if self.reg == 'l1':
                loss += self.l1_coef * np.sum(np.abs(self.weights))

            if self.reg == 'l2':
                loss += self.l2_coef * np.sum(np.square(self.weights))

            if self.reg == 'elasticnet':
                loss += self.l1_coef * np.sum(np.abs(self.weights)) + self.l2_coef * np.sum(np.square(self.weights))
            
            if i == 1 and verbose:  
                if self.metric is None:
                    print(f"start | loss: {loss}")
                else:
                    print(f"start | loss: {loss} | {self.metric}: {self._get_metric_value(y, y_pred)}")
            
            if self.sgd_sample is not None:
                if isinstance(self.sgd_sample, int):
                    sgd_sample = self.sgd_sample
                elif isinstance(self.sgd_sample, float):
                    sgd_sample = round(self.sgd_sample * n)
                    
                sample_rows_idx = random.sample(range(X.shape[0]), sgd_sample)
            
                X_hat = []
                y_hat = []
                y_pred_hat = []
                indexes_hat = []
                for j in sample_rows_idx:
                    X_hat.append(X.iloc[j])
                    y_hat.append(y.iloc[j])
                    y_pred_hat.append(y_pred.iloc[j])
                    indexes_hat.append(X.index[j])
                X_hat = pd.concat(X_hat, axis=1).T
                y_hat = pd.Series(y_hat, index= indexes_hat)
                y_pred_hat = pd.Series(y_pred_hat, index= indexes_hat)
            else:
                X_hat = X
                y_hat = y
                y_pred_hat = y_pred
            
            gradient = 2 * (y_pred_hat - y_hat).dot(X_hat) / len(X_hat)

            if self.reg == 'l1':
                gradient += self.l1_coef * np.sign(self.weights)

            if self.reg == 'l2':
                gradient += self.l2_coef * self.weights * 2

            if self.reg == 'elasticnet':
                gradient += self.l1_coef * np.sign(self.weights) + self.l2_coef * self.weights * 2            
            
            if isinstance(self.learning_rate, float):
                learning_rate = self.learning_rate
            
            if isinstance(self.learning_rate, Callable):
                learning_rate = self.learning_rate(i)
            
            self.weights -= learning_rate * gradient

            if verbose and (i) % verbose == 0 and i > 1:  
                if self.metric is None:
                    print(f"{i} | loss: {loss}")
                else:
                    print(f"{i} | loss: {loss} | {self.metric}: {self._get_metric_value(y, y_pred)}")

        y_pred = X.dot(self.weights)
        self.__metric_value = self._get_metric_value(y, y_pred)

    def get_best_score(self):
        return self.__metric_value

    def get_coef(self) -> np.ndarray:
        if self.weights is None:
            return None

        return self.weights.values[1:]

    def predict(self, X: pd.DataFrame) -> pd.Series :
        X = X.copy()
        X.insert(loc=0, column='__bias', value=1)
        return X.dot(self.weights)
                    